In [ ]:
import geopandas as gpd
from datetime import datetime

from aerial_image_detection import settings
from aerial_image_detection.utils.city_area_handler import CityAreaHandler
from aerial_image_detection.utils.pdok_downloader import PDOKDownloader

In [ ]:
buurten_gdf = CityAreaHandler().get_city_area_gdf()

In [ ]:
target_area = settings["inference"]["target_area"]
(scale, name) = next(iter(target_area.items()))

In [ ]:
download_area = buurten_gdf[buurten_gdf[f"{scale}_naam"] == name].union_all()

download_features = ["wegdeel"]
extract_bgt_functions = {
    "parkeervlakken": ["parkeervlak"],
    "ontheffinggebied": ["voetpad", "fietspad", "voetpad op trap", "voetgangersgebied"],
}

timestamp = datetime.now().strftime(format="%Y-%m-%dT%H-%M-%S")

download_folder = f"../datasets/experiments/parkeren/bgt_{timestamp}"

extension = ".gpkg"

In [ ]:
pdok_downloader = PDOKDownloader()

files = pdok_downloader.download_features_for_area(
    features=download_features,
    area = download_area,
    download_dir=download_folder,
    extract_bgt_functions=extract_bgt_functions,
    suffix=name,
    extension=extension,
)

In [ ]:
from datetime import datetime
from typing import Union

import pandas as pd

def _is_valid_at_date(row: pd.Series, valid_date: Union[str, datetime]) -> bool:
    if isinstance(valid_date, str):
        valid_date = datetime.fromisoformat(valid_date)

    if pd.isnull(row["eindRegistratie"]):
        return row["tijdstipRegistratie"] <= valid_date
    else:
        return row["tijdstipRegistratie"] <= valid_date and row["eindRegistratie"] >= valid_date

def load_and_filter_bgt(file: str, valid_at_date: str) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(file)
    gdf["tijdstipRegistratie"] = pd.to_datetime(gdf["tijdstipRegistratie"])
    gdf["eindRegistratie"] = pd.to_datetime(gdf["eindRegistratie"])
    gdf_filtered = gdf[gdf.apply(_is_valid_at_date, axis="columns", valid_date=valid_at_date)]
    return gdf_filtered

In [ ]:
filter_date = "2025-03-20"

parkeervlakken = load_and_filter_bgt(files["parkeervlakken"], filter_date)
ontheffingsgebieden = load_and_filter_bgt(files["ontheffinggebied"], filter_date)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 12))
parkeervlakken.plot(ax=ax, color="green")
ontheffingsgebieden.plot(ax=ax, color="orange")
buurten_gdf[buurten_gdf[f"{scale}_naam"] == name].boundary.plot(ax=ax, color="black")